# ==============================================================================
# 1. DESEQ + ATAC Consensus peak calling
# ==============================================================================

In [1]:
"""
End-to-end pipeline: CD133 E14 vs E18
DESeq2 -> Consensus Peaks -> Enhancer Integration -> CellOracle Base GRN -> GRN Pruning
"""

import os
import pandas as pd
import numpy as np
from importlib import reload
import pandas as pd 
import numpy as np 
import sys
import os
# get project root (two levels up from this notebook)
project_root = os.path.abspath(os.path.join(os.path.dirname('src'), '..'))
# if in notebook:
# project_root = os.path.abspath('..')   # or adjust as needed
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.pipeline as pipeline
import src.enhancer_atac as enatac
import src.grn_pruner as grnpruner
# Reload modules
reload(pipeline)
reload(enatac)
reload(grnpruner)

# ==============================================================================
# PATHS
# ==============================================================================

base_dir = "/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC"

paths = {
    # RNA-seq
    "counts": os.path.join(base_dir, "data/rna_seq/E14_ E18_LGE_cortex_seq_Counts.csv"),
    "metadata": os.path.join(base_dir, "data/rna_seq/Samples+Pooling_RNA-Seq.csv"),
    
    # ATAC-seq
    "atac_metadata": os.path.join(base_dir, "data/atac_seq/ATAC-seq/samples_clean.csv"),
    "atac_peak_dir": os.path.join(base_dir, "data/atac_seq/ATAC-seq"),
    
    # Annotations
    "tf_list": os.path.join(base_dir, "data/annotations/Mouse_TFs_Kinases_webpage-3-30-2017.xlsx"),
    "gtf": os.path.join(base_dir, "data/annotations/gencode.vM36.annotation.gtf"),
    "enhancer_files": [
        os.path.join(base_dir, "data/annotations/enhancerAtlas_neuron_cortical.txt"),
        # os.path.join(base_dir, "data/annotations/enhancerAtlas_brain_e14.5.txt"),
        os.path.join(base_dir, "data/annotations/enhancerAtlas_cortex.txt")
    ],
    
    # Integrated data (for pipeline object only)
    "overlap_df": os.path.join(base_dir, "data/integrated/overlap_annotated.tsv"),
    "chip_annotated": os.path.join(base_dir, "data/integrated/chip_annotated_filtered.tsv"),
    "atac_annotated": os.path.join(base_dir, "data/integrated/atac_annotated.tsv"),
    
    # Output paths
    "output_dir": os.path.join(base_dir, "results/cd133_e14_e18_grn"),
    "consensus_bed": os.path.join(base_dir, "results/cd133_e14_e18_grn/consensus_cd133.bed"),
    "deseq_output": os.path.join(base_dir, "results/cd133_e14_e18_grn/deseq_temporal_cd133.csv"),
    "celloracle_h5": os.path.join(base_dir, "results/cd133_e14_e18_grn/celloracle_tfinfo.h5"),
    "celloracle_parquet": os.path.join(base_dir, "results/cd133_e14_e18_grn/celloracle_base_grn.parquet"),
    "e14_grn": os.path.join(base_dir, "results/cd133_e14_e18_grn/E14_TF_network.csv"),
    "e18_grn": os.path.join(base_dir, "results/cd133_e14_e18_grn/E18_TF_network.csv")
}

# Create output directory
os.makedirs(paths["output_dir"], exist_ok=True)

exclude_samples = ['MUC9939', 'MUC9940', 'MUC9914']

print("="*80)
print("CD133 E14 vs E18 GRN PIPELINE")
print("="*80)

# ==============================================================================
# STEP 1: RNA-SEQ DIFFERENTIAL EXPRESSION (CD133 E14 vs E18)
# ==============================================================================

print("\n" + "="*80)
print("STEP 1: DIFFERENTIAL EXPRESSION ANALYSIS")
print("="*80)

nsc = pipeline.NSCAnalysis(
    counts_path=paths['counts'],
    metadata_path=paths['metadata'],
    atac_metadata_path=paths['atac_metadata'],
    overlap_df_path=paths['overlap_df'],
    chip_annotated_path=paths['chip_annotated'],
    atac_annotated_path=paths['atac_annotated'],
    tf_list_path=paths['tf_list'],
    gtf_path=paths['gtf'],
    exclude_samples=exclude_samples
)

# Filter for CD133 only
cd133_samples = nsc.metadata[nsc.metadata['Marker'] == 'Progenitors'].index.tolist()
nsc.counts = nsc.counts[[c for c in cd133_samples if c in nsc.counts.columns]]
nsc.metadata = nsc.metadata.loc[cd133_samples]

print(f"\nCD133 samples: {len(nsc.metadata)}")
print(nsc.metadata[['Stage', 'Region', 'Marker']].value_counts())

# Run DESeq2
deseq_results = nsc.run_deseq(group1='E14', group2='E18', group_col='Stage')

# Save DESeq results
deseq_results.to_csv(paths['deseq_output'])
print(f"\nDESeq results saved to: {paths['deseq_output']}")

# Get DE TFs
de_tfs = deseq_results[
    (deseq_results['padj'] < 0.05) & 
    (deseq_results['log2FoldChange'].abs() > 1) &
    (deseq_results['is_TF'] == True)
]

deg_list_e14 = de_tfs[de_tfs['log2FoldChange'] > 1]['symbol'].tolist()
deg_list_e18 = de_tfs[de_tfs['log2FoldChange'] < -1]['symbol'].tolist()

print(f"\nE14-high TFs: {len(deg_list_e14)}")
print(f"E18-high TFs: {len(deg_list_e18)}")
print(f"\nE14 TFs: {deg_list_e14[:10]}...")
print(f"E18 TFs: {deg_list_e18[:10]}...")

# ==============================================================================
# STEP 2: BUILD CONSENSUS PEAKS FROM ATAC-SEQ
# ==============================================================================

print("\n" + "="*80)
print("STEP 2: CONSENSUS PEAK BUILDING")
print("="*80)

builder = enatac.ConsensusPeakBuilder(
    metadata_file=paths['atac_metadata'],
    peak_dir=paths['atac_peak_dir'],
    factor='CD133',
    conditions=['E14', 'E18']
)

consensus = builder.build_consensus()
builder.save_bed(consensus, paths['consensus_bed'])

print(f"\nConsensus peaks: {len(consensus)}")
print(f"Saved to: {paths['consensus_bed']}")

# ==============================================================================
# STEP 3: ENHANCER INTEGRATION
# ==============================================================================

print("\n" + "="*80)
print("STEP 3: ENHANCER ATLAS INTEGRATION")
print("="*80)

integrator = enatac.EnhancerIntegrator(
    enhancer_files=paths['enhancer_files'],
    from_assembly='mm9'
)

enhancers_mm39 = integrator.load_and_liftover()

# Overlap with consensus peaks
target_genes, overlaps_df = integrator.overlap_with_consensus(consensus)

# Get DE TFs with accessible enhancers
e14_tfs_with_enh = integrator.get_de_with_accessible_enhancers(target_genes, deg_list_e14)
e18_tfs_with_enh = integrator.get_de_with_accessible_enhancers(target_genes, deg_list_e18)

print(f"\nE14 TFs with accessible enhancers: {e14_tfs_with_enh}")
print(f"E18 TFs with accessible enhancers: {e18_tfs_with_enh}")

# Save enhancer overlaps
overlaps_df.to_csv(os.path.join(paths['output_dir'], 'enhancer_consensus_overlaps.csv'), index=False)


/home/users/adhal/micromamba/envs/scrna_target_idf/lib/python3.10/site-packages/sorted_nearest/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


CD133 E14 vs E18 GRN PIPELINE

STEP 1: DIFFERENTIAL EXPRESSION ANALYSIS
Loading data...

[1/9] Loading counts...
      Dropping 1 genes with NaN values
      28664 genes x 36 samples

[2/9] Loading RNA-seq metadata...
      36 samples
      Columns: ['ForeignID', 'Stage', 'Replicate', 'Region', 'Marker', 'Shuffle']

[3/9] Loading ATAC-seq metadata...
      15 samples
      Columns: ['Tissue', 'Factor', 'Condition', 'Treatment', 'Replicate', 'bamReads', 'Peaks', 'PeakCaller']

[4/9] Creating ATAC-to-RNA sample mapping...
      Mapped: 15 / 15 ATAC samples

[5/9] Removing outlier samples...
      Removed: ['MUC9939', 'MUC9940', 'MUC9914']
      Remaining: 33 samples

[6/9] Filtering low-count genes...

Filtering low-count genes
Threshold: counts >= 10 in >= 3 samples
  E14: 17483 genes pass filter
  E18: 17939 genes pass filter

Genes before: 28664
Genes after:  18187
Removed:      10477 (36.6%)

[7/9] Loading overlap_df (ChIP ∩ ATAC)...
      448802 overlaps
      Columns: ['chip_chr', 

Fitting size factors...
... done in 0.03 seconds.

Fitting dispersions...
... done in 26.85 seconds.

Fitting dispersion trend curve...
... done in 0.72 seconds.

Fitting MAP dispersions...
... done in 41.44 seconds.

Fitting LFCs...
... done in 13.69 seconds.

Calculating cook's distance...
... done in 0.04 seconds.

Replacing 14 outlier genes.

Fitting dispersions...
... done in 0.02 seconds.

Fitting MAP dispersions...
... done in 0.02 seconds.

Fitting LFCs...
... done in 0.01 seconds.

Running Wald tests...


Extracting results (E14 vs E18)...


... done in 4.46 seconds.



Log2 fold change & Wald test p-value: Stage E14 vs E18
                        baseMean  log2FoldChange     lfcSE      stat  \
ENSMUSG00000000001  11392.735011        0.163045  0.071729  2.273084   
ENSMUSG00000000088   7290.014219        0.374888  0.087763  4.271573   
ENSMUSG00000000581   2662.931320       -0.024108  0.077363 -0.311623   
ENSMUSG00000006471   2191.953433        0.199988  0.038724  5.164521   
ENSMUSG00000036019    465.437536       -0.415655  0.145974 -2.847467   
...                          ...             ...       ...       ...   
ENSMUSG00000035984    171.993688       -4.000665  0.435222 -9.192241   
ENSMUSG00000035992   1390.075479       -0.098378  0.124280 -0.791582   
ENSMUSG00000036002   2360.311704       -0.017796  0.059961 -0.296788   
ENSMUSG00000036006    606.543527       -1.622156  0.212070 -7.649142   
ENSMUSG00000036009    659.660379       -0.053090  0.080047 -0.663239   

                          pvalue          padj  
ENSMUSG00000000001  2.302114e-0

/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:657: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_sig_tf = results[sig_mask & results['is_TF']].shape[0]
/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:658: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_sig_gene = results[sig_mask & ~results['is_TF']].shape[0]
/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:660: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_up = results[sig_mask & (results['log2FoldChange'] > 0)].shape[0]
/mnt/scratch/users/adhal/CorticalNeuronFate/CellConversionNSC/src/pipeline.py:661: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  n_down = results[sig_mask & (results['log2FoldChange'] < 0)].shape[0]



DESeq results saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/deseq_temporal_cd133.csv

E14-high TFs: 39
E18-high TFs: 83

E14 TFs: ['Hmga2', 'Smad3', 'Rcor2', 'Plagl2', 'Sox3', 'Sall4', 'Lhx9', 'Prdm12', 'Lef1', 'Hmga1']...
E18 TFs: ['Csdc2', 'Creb3l2', 'Egr1', 'Etv4', 'Dbx2', 'Klf9', 'Nfatc1', 'Sox10', 'Foxo1', 'Zbtb7c']...

STEP 2: CONSENSUS PEAK BUILDING
Found 10 samples for CD133 in ['E14', 'E18']
Consensus peaks: 93492
Saved to /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/consensus_cd133.bed

Consensus peaks: 93492
Saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/consensus_cd133.bed

STEP 3: ENHANCER ATLAS INTEGRATION
Initializing liftOver from mm9 to mm39...
Loading enhancerAtlas_neuron_cortical.txt
Loading enhancerAtlas_cortex.txt
Loaded 124051 unique enhancer-gene pairs in mm9
Lifting over to mm39...
LiftOver: 124012/124051 (100.0%)


In [45]:
overlaps_df

,Chromosome,Start,End,symbol,score,Start_b,End_b
0,chr1,166082028,166088968,Ildr2,7.504045,166081475,166082121
1,chr1,166082028,166088968,Ildr2,7.504045,166083479,166083583
2,chr1,75358089,75358679,Gm15178,7.184706,75358306,75359324
3,chr1,168233508,168255428,Pbx1,6.908724,168238327,168238506
4,chr1,168233508,168255428,Pbx1,6.908724,168244539,168244936
...,...,...,...,...,...,...,...
87802,chrX,133196370,133198000,Taf7l,0.012776,133196049,133196561
87803,chrX,133196370,133198000,Taf7l,0.012776,133197558,133197705
87804,chrX,7764873,7765003,Gripap1,0.012589,7764995,7766007
87805,chrX,7765233,7765403,Gripap1,0.012589,7764995,7766007


# ==============================================================================
# 2. CELLORACLE BASE GRN CONSTRUCTION
# ==============================================================================

In [2]:
import src.base_grn as base_grn

In [17]:

import numpy as np
import pandas as pd
import seaborn as sns
import sys, os

import matplotlib.pyplot as plt
from gimmemotifs.motif import Motif,read_motifs

## 2.1. Custom motif maker

In [65]:


# # # Load TF information as a dataframe.
# # df = pd.read_table("TF_Information_all_motifs.txt")
# # df.head()
# # # All process will be done inside these function.

# # from datetime import datetime
# # import glob

# # def read_pwn_and_convert_into_list(path):
# #     # read pwn as df
# #     pwm = pd.read_csv(path, delimiter="\t")

# #     # convert into list of str
# #     li = []
# #     for i in pwm.iterrows():
# #         i = i[1].values[1:]
# #         i = "\t".join(i.astype("str")) + "\n"
# #         li.append(i)

# #     return li

# # def make_motif_file_from_cisbp_data(pwm_folder_path, tfinfo_df, species):

# #     data_ = tfinfo_df[tfinfo_df.TF_Species == species]
# #     data_name = "CisBP_ver2_" + species

# #     ## 1. Make file: motif2factors.txt

# #     # Select information
# #     df_factors = data_[["Motif_ID", "TF_Name", "MSource_Type", "TF_Status"]]
# #     df_factors = df_factors[df_factors.TF_Status != "N"]

# #     # Formatting
# #     df_factors.columns = 'Motif\tFactor\tEvidence\tCurated'.split("\t")
# #     df_factors["Curated"] = [{"D": "Y", "I": "N"}[i] for i in df_factors["Curated"]]
# #     df_factors = df_factors.sort_values(by="Motif")

# #     ## 2. Make file: pfm file
# #     comments = f"# CIS-BP motif database (v2.0), retrieved by Celloracle\n"
# #     comments += "# Retrieved from: http://cisbp.ccbr.utoronto.ca/data/2.00/DataFiles/Bulk_downloads/EntireDataset/PWMs.zip\n"
# #     comments += f"#Date: {datetime.now().ctime()}\n"

# #     # Get list of motif name
# #     paths_pwm = glob.glob(os.path.join(pwm_folder_path, "*.txt"))
# #     paths_pwm.sort()
# #     motif_names = [path.split("/")[-1].replace(".txt", "") for path in paths_pwm]
# #     motifs = np.intersect1d(motif_names, df_factors.Motif.unique())

# #     print(motifs.shape)

# #     # Intersect motif information with pwm information
# #     df_factors = df_factors[df_factors.Motif.isin(motifs)]

# #     # Load, convert, and save pwm info
# #     output = data_name + ".pfm"

# #     motifs_non_zero = []
# #     with open(output, "w") as f:

# #         for motif_name in motifs:

# #             path = os.path.join(pwm_folder_path, motif_name + ".txt")
# #             pwm = read_pwn_and_convert_into_list(path=path) # Load and convert
# #             if pwm:
# #                 motifs_non_zero.append(motif_name)
# #                 pwm = [f">{motif_name}\n"] + pwm
# #                 for i in pwm: # Save pfm
# #                     f.write(i)


# #     # Intersect motif information with pwm information
# #     df_factors = df_factors[df_factors.Motif.isin(motifs_non_zero)]

# #     # Save factor info
# #     df_factors.to_csv(f'{data_name}.motif2factors.txt', sep='\t', index=False)

# #     print(df_factors.shape, len(motifs_non_zero))
# species = 'Mus_musculus'
# make_motif_file_from_cisbp_data(pwm_folder_path="/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/annotations/pwms", tfinfo_df=df, species=species)

## 2.2. Base GRN using CO

In [ ]:
print("\n" + "="*80)
print("STEP 4: CELLORACLE BASE GRN CONSTRUCTION")
print("="*80)


# Initialize CellOracle
co = base_grn.GRNCo(
    bed_path=paths['consensus_bed'],
    ref_genome='mm39',
    genomes_dir=None
)

# Step 1: Load BED
print("\nLoading consensus peaks...")
co.load_bed()

# Step 2: Annotate TSS
print("Annotating TSS...")
co.annotate_tss()

# Step 3: Ensure genome
print("Checking genome installation...")
co.ensure_genome()

# Step 4: Scan motifs
print("Scanning TF motifs (this takes time)...")
co.scan_motifs(fpr=0.02, verbose=True)

# Step 5: Filter motifs
print("Filtering motifs...")
co.filter_motifs(score_threshold=10)




STEP 4: CELLORACLE BASE GRN CONSTRUCTION

Loading consensus peaks...
Annotating TSS...
que bed peaks: 93492
tss peaks in que: 36221
Checking genome installation...
Scanning TF motifs (this takes time)...
No motif data entered. Loading default motifs for your species ...
 Default motif for vertebrate: gimme.vertebrate.v5.0. 
 For more information, please see https://gimmemotifs.readthedocs.io/en/master/overview.html 

Initiating scanner... 



2025-12-16 14:31:56,925 - DEBUG - using background: genome mm39 with size 200


Calculating FPR-based threshold. This step may take substantial time when you load a new ref-genome. It will be done quicker on the second time. 

Motif scan started .. It may take long time.



Scanning:   0%|          | 0/17436 [00:00<?, ? sequences/s]

Filtering motifs...
Filtering finished: 9877109 -> 1839134
1. Converting scanned results into one-hot encoded dataframe.


  0%|          | 0/17400 [00:00<?, ?it/s]

2. Converting results into dictionaries.


  0%|          | 0/18608 [00:00<?, ?it/s]

  0%|          | 0/1090 [00:00<?, ?it/s]

In [20]:
co.tss_annotated

,peak_id,gene_short_name
0,chr1_3740869_3742728,Xkr4
1,chr1_3584982_3585271,Gm37329
2,chr1_4855424_4856438,Mrpl15
3,chr1_4855424_4856438,Mrpl15
4,chr1_4855424_4856438,Mrpl15
...,...,...
36216,chrY_90795052_90796359,Gm47283
36217,chrY_90795052_90796359,Gm47283
36218,chrY_90795052_90796359,Gm47283
36219,chrY_90795052_90796359,Gm47283


## 2.3. Adding TSS distance to annotated peaks

In [28]:
import gtfparse
def add_tss_distance(base_grn_annotation, gtf_file):
    """
    Add distance_to_tss column to your CellOracle peak annotation.
    
    Args:
        base_grn_annotation: Your df with columns [peak_id, gene_short_name]
        gtf_file: Path to GTF file
    
    Returns:
        DataFrame with added distance_to_tss column
    """
    import pandas as pd
    import numpy as np
    import gtfparse
    
    # 1. Parse peak coordinates from peak_id
    base_grn_annotation = base_grn_annotation.copy()
    base_grn_annotation[['chr', 'start', 'end']] = base_grn_annotation['peak_id'].str.extract(
        r'(chr[^_]+)_(\d+)_(\d+)'
    )
    base_grn_annotation['start'] = base_grn_annotation['start'].astype(int)
    base_grn_annotation['end'] = base_grn_annotation['end'].astype(int)
    base_grn_annotation['peak_center'] = (base_grn_annotation['start'] + base_grn_annotation['end']) / 2
    
    # 2. Load gene TSS coordinates
    gtf = gtfparse.read_gtf(gtf_file)
    
    # Convert Polars to Pandas if needed
    if hasattr(gtf, 'to_pandas'):
        gtf = gtf.to_pandas()
    
    # Get TSS per gene - use FILTER for Polars, [] for Pandas
    tss_df = gtf[gtf['feature'] == 'transcript'].copy()
    tss_df['tss'] = np.where(tss_df['strand'] == '+', 
                              tss_df['start'], 
                              tss_df['end'])
    
    # Keep one TSS per gene
    tss_per_gene = tss_df.groupby('gene_name').agg({
        'seqname': 'first',
        'tss': 'first'
    }).reset_index()
    tss_per_gene.columns = ['gene_short_name', 'gene_chr', 'gene_tss']
    
    # 3. Merge and calculate distance
    merged = base_grn_annotation.merge(
        tss_per_gene, 
        on='gene_short_name', 
        how='left'
    )
    
    # Distance = |peak_center - TSS|
    merged['distance_to_tss'] = np.abs(merged['peak_center'] - merged['gene_tss'])
    
    # Handle unmatched genes
    merged['distance_to_tss'].fillna(1e6, inplace=True)
    
    return merged[['peak_id', 'gene_short_name', 'chr', 'start', 'end', 'distance_to_tss']]




In [29]:
def add_tss_distance_pyranges(base_grn_annotation, gtf_file):
    """Using PyRanges for genomic distance calculation."""
    import pandas as pd
    import numpy as np
    import pyranges as pr
    
    # 1. Parse peak coordinates
    base_grn_annotation = base_grn_annotation.copy()
    base_grn_annotation[['chr', 'start', 'end']] = base_grn_annotation['peak_id'].str.extract(
        r'(chr[^_]+)_(\d+)_(\d+)'
    )
    base_grn_annotation['start'] = base_grn_annotation['start'].astype(int)
    base_grn_annotation['end'] = base_grn_annotation['end'].astype(int)
    
    # 2. Load GTF with PyRanges
    gtf = pr.read_gtf(gtf_file)
    
    # Get transcripts only
    transcripts = gtf[gtf.Feature == "transcript"]
    
    # Calculate TSS based on strand
    tss_df = transcripts.df
    tss_df['tss'] = np.where(tss_df['Strand'] == '+', 
                              tss_df['Start'], 
                              tss_df['End'])
    
    # One TSS per gene
    tss_per_gene = tss_df.groupby('gene_name').agg({
        'Chromosome': 'first',
        'tss': 'first'
    }).reset_index()
    tss_per_gene.columns = ['gene_short_name', 'gene_chr', 'gene_tss']
    
    # 3. Merge
    merged = base_grn_annotation.merge(tss_per_gene, on='gene_short_name', how='left')
    
    # Calculate distance
    merged['peak_center'] = (merged['start'] + merged['end']) / 2
    merged['distance_to_tss'] = np.abs(merged['peak_center'] - merged['gene_tss'])
    merged['distance_to_tss'].fillna(1e6, inplace=True)
    
    return merged[['peak_id', 'gene_short_name', 'chr', 'start', 'end', 'distance_to_tss']]

# Usage:
peak_annot = add_tss_distance_pyranges(
    base_grn_annotation=co.tss_annotated,
    gtf_file=paths['gtf']
)


# ==============================================================================
# 3. GRN PRUNING
# ==============================================================================

In [67]:
paths['tf_list']

'/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/data/annotations/Mouse_TFs_Kinases_webpage-3-30-2017.xlsx'

In [71]:
tf_df = pd.read_excel(paths['tf_list'])
tf_df = tf_df.dropna()
tf_df = tf_df.iloc[1:, :]
tf_df.columns  = ['Symbol', 'Annotation', 'Family', 'Ensembl_ID', 'uniprot_ID']

In [72]:
tf_df

,Symbol,Annotation,Family,Ensembl_ID,uniprot_ID
6,Adnp,activity-dependent neuroprotective protein,Homeobox,ENSMUSG00000051149,Q9Z103
7,Aebp2,AE binding protein 2,zf-C2H2,ENSMUSG00000030232,Q9Z248
8,Aff1,"AF4/FMR2 family, member 1",AF-4,ENSMUSG00000029313,O88573
9,Aff2,"AF4/FMR2 family, member 2",AF-4,ENSMUSG00000031189,O55112
10,Aff3,"AF4/FMR2 family, member 3",AF-4,ENSMUSG00000037138,P51827
...,...,...,...,...,...
1483,9130019O22Rik,RIKEN cDNA 9130019O22 gene,zf-C2H2,ENSMUSG00000030823,-
1484,9130023H24Rik,RIKEN cDNA 9130023H24 gene,zf-C2H2,ENSMUSG00000062944,-
1485,9830147E19Rik,RIKEN cDNA 9830147E19 gene,zf-C2H2,ENSMUSG00000074158,-
1486,A430033K04Rik,RIKEN cDNA A430033K04 gene,zf-C2H2,ENSMUSG00000056014,-


In [6]:
paths['celloracle_h5'] = '/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/cd133.celloracle.tfinfo'

In [ ]:
reload(grnpruner)
# Step 6: Save outputs
print("Saving CellOracle outputs...")
co.save_tfinfo(paths['celloracle_h5'])
base_grn_df = co.save_dataframe(paths['celloracle_parquet'])

print(f"\nBase GRN saved to: {paths['celloracle_parquet']}")
print(f"Base GRN shape: {base_grn_df.shape}")
print(f"TFs in base GRN: {base_grn_df.shape[1] - 2}")  # -2 for peak_id and gene columns

# ==============================================================================
# STEP 5: GRN PRUNING (CORRELATION + DE FILTERING)
# ==============================================================================

print("\n" + "="*80)
print("STEP 5: GRN PRUNING")
print("="*80)

# Get all TFs (E14 + E18 DE TFs)
all_tf_list = tf_df['Symbol'].tolist()
print(f"\nTotal DE TFs for pruning: {len(all_tf_list)}")

# Initialize pruner
pruner = grnpruner.GRNPruner(
    pkn_file=paths['celloracle_parquet'],
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    tf_list=all_tf_list,
    lfc_threshold=1.0,
    corr_threshold=0.7,
    qval_threshold=0.05
)

# Build stage-specific networks
e14_grn, e18_grn = pruner.build_networks()

# Save networks
e14_grn.to_csv(paths['e14_grn'], index=False)
e18_grn.to_csv(paths['e18_grn'], index=False)

print(f"\nE14 network saved to: {paths['e14_grn']}")
print(f"E18 network saved to: {paths['e18_grn']}")

# ==============================================================================
# STEP 6: SUMMARY
# ==============================================================================

print("\n" + "="*80)
print("PIPELINE COMPLETE - SUMMARY")
print("="*80)

summary = pd.DataFrame({
    'Step': [
        '1. DESeq2',
        '2. Consensus Peaks',
        '3. Enhancer Integration',
        '4. CellOracle Base GRN',
        '5. E14 TF Network',
        '6. E18 TF Network'
    ],
    'Count': [
        f"{len(de_tfs)} DE TFs",
        f"{len(consensus)} peaks",
        f"{len(target_genes)} genes with accessible enhancers",
        f"{base_grn_df.shape[1] - 2} TFs in base GRN",
        f"{len(e14_grn)} edges, {e14_grn['TF'].nunique()} TFs",
        f"{len(e18_grn)} edges, {e18_grn['TF'].nunique()} TFs"
    ]
})

print(summary.to_markdown(index=False))

print("\n" + "="*80)
print("OUTPUT FILES:")
print("="*80)
for key, path in paths.items():
    if 'output' in key or any(x in key for x in ['consensus', 'deseq', 'celloracle', 'grn']):
        print(f"  {key}: {path}")

print("\n" + "="*80)
print("NETWORK COMPARISON:")
print("="*80)

e14_edges = set(zip(e14_grn['TF'], e14_grn['target']))
e18_edges = set(zip(e18_grn['TF'], e18_grn['target']))

shared_edges = e14_edges & e18_edges
e14_specific = e14_edges - e18_edges
e18_specific = e18_edges - e14_edges

print(f"\nShared edges: {len(shared_edges)}")
print(f"E14-specific edges: {len(e14_specific)}")
print(f"E18-specific edges: {len(e18_specific)}")

print("\nTop 10 E14 hub TFs:")
print(e14_grn['TF'].value_counts().head(10))

print("\nTop 10 E18 hub TFs:")
print(e18_grn['TF'].value_counts().head(10))

Saving CellOracle outputs...

Base GRN saved to: /mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/celloracle_base_grn.parquet
Base GRN shape: (20783, 1092)
TFs in base GRN: 1090

STEP 5: GRN PRUNING

Total DE TFs for pruning: 1482
Loading data...
PKN: 20783 peaks, TF list: 1473 TFs

=== Converting PKN ===


# ==============================================================================
# 4 GRN Plotting
# ==============================================================================

In [39]:
def plot_hierarchical_network(grn_df, stage_name, output_file, 
                              break_cycles=True, figsize=(30, 20)):
    """
    Plot hierarchical TF network.
    
    Args:
        grn_df: DataFrame with columns [TF, target, correlation, ...]
        stage_name: Name for title (e.g., 'E14', 'E18')
        output_file: Path to save PNG
        break_cycles: If True, break cycles for true topological hierarchy
        figsize: Figure size
    """
    import networkx as nx
    import matplotlib.pyplot as plt
    
    # Remove self-loops
    grn_clean = grn_df[grn_df['TF'] != grn_df['target']].copy()
    print(f"\n{stage_name}: Removed {len(grn_df) - len(grn_clean)} self-loops")
    
    # Build graph
    G = nx.from_pandas_edgelist(grn_clean, source='TF', target='target', 
                                 create_using=nx.DiGraph())
    
    print(f"{stage_name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    
    # Break cycles if requested
    if break_cycles:
        G_plot = G.copy()
        removed = 0
        while not nx.is_directed_acyclic_graph(G_plot):
            try:
                cycle = nx.find_cycle(G_plot)
                G_plot.remove_edge(cycle[0][0], cycle[0][1])
                removed += 1
            except nx.NetworkXNoCycle:
                break
        print(f"{stage_name}: Removed {removed} edges to break cycles")
        layout_type = "True Topological Hierarchy"
    else:
        G_plot = G
        layout_type = "Pseudo-Hierarchy (by degree)"
    
    # Calculate layout
    plt.figure(figsize=figsize)
    
    try:
        # Try topological layout
        layers = {}
        for i, nodes in enumerate(nx.topological_generations(G_plot)):
            for node in nodes:
                layers[node] = i
        
        layer_nodes = {}
        for node, layer in layers.items():
            if layer not in layer_nodes:
                layer_nodes[layer] = []
            layer_nodes[layer].append(node)
        
        pos = {}
        for layer, nodes in layer_nodes.items():
            n = len(nodes)
            for i, node in enumerate(nodes):
                x = (i - n/2) * 2
                y = -layer * 3
                pos[node] = (x, y)
        
        print(f"{stage_name}: Using topological layout")
        
    except nx.NetworkXUnfeasible:
        # Fallback: pseudo-hierarchy
        print(f"{stage_name}: Still has cycles, using pseudo-hierarchy")
        in_degrees = dict(G_plot.in_degree())
        out_degrees = dict(G_plot.out_degree())
        
        layers = {}
        for node in G_plot.nodes():
            layers[node] = out_degrees[node] - in_degrees[node]
        
        max_score = max(layers.values()) if layers.values() else 1
        min_score = min(layers.values()) if layers.values() else 0
        
        for node in layers:
            if max_score != min_score:
                layers[node] = int(5 * (layers[node] - min_score) / (max_score - min_score))
            else:
                layers[node] = 0
        
        layer_nodes = {}
        for node, layer in layers.items():
            if layer not in layer_nodes:
                layer_nodes[layer] = []
            layer_nodes[layer].append(node)
        
        pos = {}
        for layer, nodes in layer_nodes.items():
            n = len(nodes)
            for i, node in enumerate(nodes):
                x = (i - n/2) * 2
                y = layer * 3
                pos[node] = (x, y)
        
        layout_type = "Pseudo-Hierarchy (by degree)"
    
    # Adjust font size based on network size
    if G_plot.number_of_nodes() > 100:
        font_size = 6
        node_size = 300
    elif G_plot.number_of_nodes() > 50:
        font_size = 8
        node_size = 500
    else:
        font_size = 10
        node_size = 800
    
    # Draw
    nx.draw_networkx_nodes(G_plot, pos, node_size=node_size, node_color='lightblue', alpha=0.8)
    nx.draw_networkx_edges(G_plot, pos, alpha=0.3, arrows=True, arrowsize=10, 
                           edge_color='gray', width=1)
    nx.draw_networkx_labels(G_plot, pos, font_size=font_size, font_weight='bold')
    
    title = f'{stage_name} TF Network - {layout_type}\n({G_plot.number_of_nodes()} nodes, {G_plot.number_of_edges()} edges)'
    plt.title(title, fontsize=18)
    plt.axis('off')
    plt.tight_layout()
    
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    
    print(f"{stage_name}: Saved to {output_file}\n")
    
    return G_plot


# Usage - full networks:
e14_graph = plot_hierarchical_network(
    grn_df=e14_grn,
    stage_name='E14',
    output_file='e14_network_full_hierarchical.png',
    break_cycles=True,
    figsize=(30, 20)  # Large canvas for full network
)

e18_graph = plot_hierarchical_network(
    grn_df=e18_grn,
    stage_name='E18',
    output_file='e18_network_full_hierarchical.png',
    break_cycles=True,
    figsize=(30, 20)
)


E14: Removed 0 self-loops
E14: 29 nodes, 50 edges
E14: Removed 1 edges to break cycles
E14: Using topological layout
E14: Saved to e14_network_full_hierarchical.png


E18: Removed 0 self-loops
E18: 37 nodes, 59 edges
E18: Removed 5 edges to break cycles
E18: Using topological layout
E18: Saved to e18_network_full_hierarchical.png



In [80]:
def assign_hierarchical_layers_genomic_hybrid(grn_df, base_grn_motif, peak_annot, enhancer_overlap_df):
    """
    Hybrid approach with FIXED case handling.
    """
    import networkx as nx
    import numpy as np
    import pandas as pd
    
    # Extract enhancer peaks
    enhancer_overlap_df['peak_id'] = (
        enhancer_overlap_df['Chromosome'].astype(str) + '_' + 
        enhancer_overlap_df['Start_b'].astype(str) + '_' + 
        enhancer_overlap_df['End_b'].astype(str)
    )
    enhancer_peak_ids = set(enhancer_overlap_df['peak_id'].unique())
    
    # Classify peaks
    def classify_peak(row):
        if row['peak_id'] in enhancer_peak_ids:
            return 'enhancer_db'
        elif row['distance_to_tss'] < 1000:
            return 'promoter'
        elif row['distance_to_tss'] < 10000:
            return 'proximal'
        elif row['distance_to_tss'] < 50000:
            return 'distal'
        else:
            return 'enhancer_distal'
    
    peak_annot['peak_class'] = peak_annot.apply(classify_peak, axis=1)
    peak_annot['is_enhancer_like'] = peak_annot['peak_class'].isin([
        'enhancer_db', 'distal', 'enhancer_distal'
    ])
    
    print(f"\n=== Peak Classification (Hybrid) ===")
    print(f"Total unique peaks: {len(peak_annot)}")
    print(f"EnhancerDB: {len(peak_annot[peak_annot['peak_class'] == 'enhancer_db'])} ({len(peak_annot[peak_annot['peak_class'] == 'enhancer_db'])/len(peak_annot)*100:.1f}%)")
    print(f"Promoter: {len(peak_annot[peak_annot['peak_class'] == 'promoter'])} ({len(peak_annot[peak_annot['peak_class'] == 'promoter'])/len(peak_annot)*100:.1f}%)")
    print(f"Combined enhancer-like: {peak_annot['is_enhancer_like'].sum()} ({peak_annot['is_enhancer_like'].sum()/len(peak_annot)*100:.1f}%)")
    
    # Standardize TF names to match motif matrix
    grn_df = grn_df.copy()
    grn_df['TF'] = grn_df['TF'].str.capitalize()
    grn_df['target'] = grn_df['target'].str.capitalize()
    
    # FIXED: Case-insensitive mapping that returns original column name
    motif_col_map = {col.lower(): col for col in base_grn_motif.columns}
    
    # Calculate TF binding profiles
    tf_profiles = {}
    
    print(f"\n=== TF Binding Profiles ===")
    
    for tf in sorted(set(grn_df['TF'].unique()) | set(grn_df['target'].unique())):
        tf_lower = tf.lower()
        
        # FIXED: Use case-insensitive lookup
        if tf_lower not in motif_col_map:
            print(f"{tf}: NOT FOUND in motif matrix")
            tf_profiles[tf] = {
                'n_peaks': 0, 'n_enhancer_db': 0, 'n_distal': 0,
                'n_enhancer_like': 0, 'n_promoter': 0,
                'enhancer_like_ratio': 0, 'median_distance': 0,
                'upstream_score': 0
            }
            continue
        
        # Get actual column name (with correct case)
        motif_col = motif_col_map[tf_lower]
        
        # Get peaks where this TF binds
        tf_peaks = base_grn_motif[base_grn_motif[motif_col] == 1].index.tolist()
        
        if len(tf_peaks) == 0:
            print(f"{tf}: No binding peaks (motif found but no sites)")
            tf_profiles[tf] = {
                'n_peaks': 0, 'n_enhancer_db': 0, 'n_distal': 0,
                'n_enhancer_like': 0, 'n_promoter': 0,
                'enhancer_like_ratio': 0, 'median_distance': 0,
                'upstream_score': 0
            }
            continue
        
        tf_peak_info = peak_annot[peak_annot['peak_id'].isin(tf_peaks)]
        
        if len(tf_peak_info) == 0:
            print(f"{tf}: Peaks not in annotation")
            continue
        
        # Count by classification
        n_total = len(tf_peak_info)
        n_enhancer_db = len(tf_peak_info[tf_peak_info['peak_class'] == 'enhancer_db'])
        n_promoter = len(tf_peak_info[tf_peak_info['peak_class'] == 'promoter'])
        n_proximal = len(tf_peak_info[tf_peak_info['peak_class'] == 'proximal'])
        n_distal = len(tf_peak_info[tf_peak_info['peak_class'] == 'distal'])
        n_enhancer_distal = len(tf_peak_info[tf_peak_info['peak_class'] == 'enhancer_distal'])
        
        n_enhancer_like = tf_peak_info['is_enhancer_like'].sum()
        enhancer_like_ratio = n_enhancer_like / n_total
        
        median_dist = tf_peak_info['distance_to_tss'].median()
        
        upstream_score = (enhancer_like_ratio * 1000) + (median_dist / 100)
        
        tf_profiles[tf] = {
            'n_peaks': n_total,
            'n_enhancer_db': n_enhancer_db,
            'n_promoter': n_promoter,
            'n_proximal': n_proximal,
            'n_distal': n_distal,
            'n_enhancer_distal': n_enhancer_distal,
            'n_enhancer_like': n_enhancer_like,
            'enhancer_like_ratio': enhancer_like_ratio,
            'median_distance': median_dist,
            'upstream_score': upstream_score
        }
        
        print(f"{tf}: {n_total} peaks → {n_enhancer_like} enh-like ({enhancer_like_ratio:.1%}), median_dist={median_dist:.0f}bp")
    
    # Build graph
    G = nx.DiGraph()
    for _, row in grn_df.iterrows():
        source_score = tf_profiles.get(row['TF'], {}).get('upstream_score', 0)
        target_score = tf_profiles.get(row['target'], {}).get('upstream_score', 0)
        edge_weight = source_score - target_score
        G.add_edge(row['TF'], row['target'], weight=edge_weight, correlation=row['correlation'])
    
    # Break cycles
    G_dag = G.copy()
    removed_edges = []
    
    while not nx.is_directed_acyclic_graph(G_dag):
        try:
            cycle = nx.find_cycle(G_dag)
            min_weight = float('inf')
            edge_to_remove = None
            
            for u, v in cycle:
                weight = G_dag[u][v]['weight']
                if weight < min_weight:
                    min_weight = weight
                    edge_to_remove = (u, v)
            
            if edge_to_remove:
                removed_edges.append(edge_to_remove)
                G_dag.remove_edge(*edge_to_remove)
        except nx.NetworkXNoCycle:
            break
    
    print(f"\n=== Cycle Breaking ===")
    print(f"Removed {len(removed_edges)} edges:")
    for u, v in removed_edges:
        u_prof = tf_profiles.get(u, {})
        v_prof = tf_profiles.get(v, {})
        print(f"  {u} ({u_prof.get('enhancer_like_ratio', 0):.1%} enh-like) → "
              f"{v} ({v_prof.get('enhancer_like_ratio', 0):.1%} enh-like)")
    
    # Assign layers
    layers = {}
    for i, nodes in enumerate(nx.topological_generations(G_dag)):
        for node in nodes:
            layers[node] = i
    
    print(f"\n=== Layer Summary ===")
    layer_nodes = {}
    for node, layer in layers.items():
        if layer not in layer_nodes:
            layer_nodes[layer] = []
        layer_nodes[layer].append(node)
    
    for layer in sorted(layer_nodes.keys()):
        nodes = layer_nodes[layer]
        nodes_with_data = [n for n in nodes if tf_profiles.get(n, {}).get('n_peaks', 0) > 0]
        
        if nodes_with_data:
            avg_enh = np.mean([tf_profiles.get(n, {}).get('enhancer_like_ratio', 0) for n in nodes_with_data])
        else:
            avg_enh = 0
        
        print(f"\nLayer {layer}: {len(nodes)} TFs, avg enhancer-like ratio = {avg_enh:.1%}")
        for tf in sorted(nodes, key=lambda x: tf_profiles.get(x, {}).get('enhancer_like_ratio', 0), reverse=True):
            prof = tf_profiles.get(tf, {})
            if prof.get('n_peaks', 0) > 0:
                print(f"  {tf}: {prof.get('n_enhancer_like', 0)}/{prof.get('n_peaks', 0)} enh-like "
                      f"({prof.get('enhancer_like_ratio', 0):.1%}) "
                      f"[DB:{prof.get('n_enhancer_db', 0)}, Distal:{prof.get('n_distal', 0)+prof.get('n_enhancer_distal', 0)}]")
            else:
                print(f"  {tf}: NO DATA")
    
    return layers, tf_profiles
# Keep only unique peaks (first occurrence)
peak_annot_unique = peak_annot.drop_duplicates(subset='peak_id', keep='first')
print(f"Unique peaks in peak_annot: {len(peak_annot_unique)}")
# Set index properly
base_grn_motif_indexed = base_grn_df.set_index('peak_id')
base_grn_motif_indexed = base_grn_motif_indexed.drop(columns=['gene_short_name'])

Unique peaks in peak_annot: 17436


In [81]:
# Run with fixed case handling
layers_e14, profiles_e14 = assign_hierarchical_layers_genomic_hybrid(
    grn_df=e14_grn,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    enhancer_overlap_df=overlaps_df
)

# Run with fixed case handling
layers_e18, profiles_e18 = assign_hierarchical_layers_genomic_hybrid(
    grn_df=e18_grn,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    enhancer_overlap_df=overlaps_df
)


=== Peak Classification (Hybrid) ===
Total unique peaks: 17436
EnhancerDB: 6309 (36.2%)
Promoter: 8565 (49.1%)
Combined enhancer-like: 7817 (44.8%)

=== TF Binding Profiles ===
Dmrt2: 109 peaks → 50 enh-like (45.9%), median_dist=275bp
Dmrt3: 69 peaks → 26 enh-like (37.7%), median_dist=246bp
E2f2: 13386 peaks → 6615 enh-like (49.4%), median_dist=200bp
E2f3: 13051 peaks → 6494 enh-like (49.8%), median_dist=202bp
Hmga1: 109 peaks → 47 enh-like (43.1%), median_dist=313bp
Lef1: 205 peaks → 102 enh-like (49.8%), median_dist=288bp
Neurog1: 1297 peaks → 623 enh-like (48.0%), median_dist=251bp
Nhlh1: 5535 peaks → 2724 enh-like (49.2%), median_dist=234bp
Nr0b1: 1370 peaks → 756 enh-like (55.2%), median_dist=287bp
Plagl2: NOT FOUND in motif matrix
Sall4: 4213 peaks → 2230 enh-like (52.9%), median_dist=202bp
Tcf7l1: 184 peaks → 93 enh-like (50.5%), median_dist=307bp
Zfp599: NOT FOUND in motif matrix

=== Cycle Breaking ===
Removed 1 edges:
  E2f2 (49.4% enh-like) → E2f3 (49.8% enh-like)

=== Laye

In [82]:
# Analyze chromosome distribution in the GRN

def analyze_chromosome_bias(grn_df, base_grn_motif, peak_annot):
    """
    Check if TF-target links are chromosome-biased.
    """
    import pandas as pd
    
    print("=== Chromosome Bias Analysis ===\n")
    
    # 1. For each TF-target pair, identify which chromosomes the binding occurs on
    
    tf_target_chr = []
    
    for _, row in grn_df.iterrows():
        tf = row['TF']
        target = row['target']
        
        # Get TF's binding peaks
        tf_col = [col for col in base_grn_motif.columns if col.lower() == tf.lower()]
        if not tf_col:
            continue
        tf_col = tf_col[0]
        
        tf_peaks = base_grn_motif[base_grn_motif[tf_col] == 1].index.tolist()
        
        # Filter to peaks linked to target gene
        tf_to_target_peaks = peak_annot[
            (peak_annot['peak_id'].isin(tf_peaks)) &
            (peak_annot['gene_short_name'] == target)
        ]
        
        if len(tf_to_target_peaks) == 0:
            continue
        
        # Get chromosomes where this regulation occurs
        chromosomes = tf_to_target_peaks['chr'].unique().tolist()
        
        tf_target_chr.append({
            'TF': tf,
            'target': target,
            'n_peaks': len(tf_to_target_peaks),
            'chromosomes': chromosomes,
            'n_chr': len(chromosomes),
            'primary_chr': tf_to_target_peaks['chr'].value_counts().index[0]
        })
    
    chr_df = pd.DataFrame(tf_target_chr)
    
    print(f"Total TF-target pairs analyzed: {len(chr_df)}")
    print(f"\nChromosome distribution:")
    print(f"  Single chromosome: {len(chr_df[chr_df['n_chr'] == 1])} ({len(chr_df[chr_df['n_chr'] == 1])/len(chr_df)*100:.1f}%)")
    print(f"  Multiple chromosomes: {len(chr_df[chr_df['n_chr'] > 1])} ({len(chr_df[chr_df['n_chr'] > 1])/len(chr_df)*100:.1f}%)")
    
    # 2. Check if TF's own location matters
    
    print(f"\n=== TF and Target Gene Chromosome Locations ===\n")
    
    # Get gene locations from peak_annot
    gene_chr = peak_annot.groupby('gene_short_name')['chr'].first().to_dict()
    
    cis_trans = []
    
    for _, row in grn_df.iterrows():
        tf = row['TF']
        target = row['target']
        
        tf_chr = gene_chr.get(tf)
        target_chr = gene_chr.get(target)
        
        if tf_chr and target_chr:
            cis_trans.append({
                'TF': tf,
                'target': target,
                'TF_chr': tf_chr,
                'target_chr': target_chr,
                'is_cis': tf_chr == target_chr
            })
    
    cis_trans_df = pd.DataFrame(cis_trans)
    
    print(f"TF-target pairs with known locations: {len(cis_trans_df)}")
    print(f"\nCis vs Trans regulation:")
    print(f"  Cis (same chromosome): {cis_trans_df['is_cis'].sum()} ({cis_trans_df['is_cis'].sum()/len(cis_trans_df)*100:.1f}%)")
    print(f"  Trans (different chr): {(~cis_trans_df['is_cis']).sum()} ({(~cis_trans_df['is_cis']).sum()/len(cis_trans_df)*100:.1f}%)")
    
    # 3. Check if certain chromosomes are over-represented
    
    print(f"\n=== Chromosome Representation in Regulatory Links ===\n")
    
    all_chr_links = []
    for chrs in chr_df['chromosomes']:
        all_chr_links.extend(chrs)
    
    chr_counts = pd.Series(all_chr_links).value_counts()
    print("Top 10 chromosomes by regulatory links:")
    print(chr_counts.head(10))
    
    # 4. Identify cross-chromosome regulatory chains
    
    print(f"\n=== Cross-Chromosome Regulatory Chains ===\n")
    
    # Example: TF1 (chr1) → TF2 (chr2) → TF3 (chr3)
    
    for layer in range(3):
        layer_tfs = [tf for tf, l in layers_e14.items() if l == layer]
        
        if len(layer_tfs) == 0:
            continue
        
        print(f"\nLayer {layer} → Layer {layer+1} transitions:")
        
        for tf in layer_tfs[:5]:  # First 5 as example
            tf_chr = gene_chr.get(tf)
            targets = grn_df[grn_df['TF'] == tf]['target'].tolist()
            
            for target in targets[:3]:  # First 3 targets
                target_chr = gene_chr.get(target)
                target_layer = layers_e14.get(target)
                
                if tf_chr and target_chr:
                    trans_label = "TRANS" if tf_chr != target_chr else "cis"
                    print(f"  {tf} ({tf_chr}) → {target} ({target_chr}) [{trans_label}]")
    
    return chr_df, cis_trans_df


# Run analysis
chr_df, cis_trans_df = analyze_chromosome_bias(
    grn_df=e14_grn,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique
)

=== Chromosome Bias Analysis ===

Total TF-target pairs analyzed: 11

Chromosome distribution:
  Single chromosome: 11 (100.0%)
  Multiple chromosomes: 0 (0.0%)

=== TF and Target Gene Chromosome Locations ===

TF-target pairs with known locations: 11

Cis vs Trans regulation:
  Cis (same chromosome): 0 (0.0%)
  Trans (different chr): 11 (100.0%)

=== Chromosome Representation in Regulatory Links ===

Top 10 chromosomes by regulatory links:
chr13    2
chr17    2
chr19    2
chr2     2
chr4     1
chr6     1
chr9     1
Name: count, dtype: int64

=== Cross-Chromosome Regulatory Chains ===


Layer 0 → Layer 1 transitions:
  E2f3 (chr13) → Hmga1 (chr17) [TRANS]
  E2f3 (chr13) → Plagl2 (chr2) [TRANS]
  E2f3 (chr13) → E2f2 (chr4) [TRANS]
  Nhlh1 (chr1) → Neurog1 (chr13) [TRANS]
  Nhlh1 (chr1) → Dmrt3 (chr19) [TRANS]
  Nhlh1 (chr1) → Zfp599 (chr9) [TRANS]
  Sall4 (chr2) → Dmrt3 (chr19) [TRANS]
  Sall4 (chr2) → Tcf7l1 (chr6) [TRANS]

Layer 1 → Layer 2 transitions:
  E2f2 (chr4) → E2f3 (chr13) [T

In [83]:
# Deep dive: Are peaks near the TARGET gene or spread genome-wide?

def analyze_peak_target_proximity(grn_df, base_grn_motif, peak_annot):
    """
    Check if TF binding peaks are near target gene or genome-wide.
    """
    import pandas as pd
    import numpy as np
    
    print("=== Peak-to-Target Proximity Analysis ===\n")
    
    proximity_data = []
    
    for _, row in grn_df.iterrows():
        tf = row['TF']
        target = row['target']
        
        # Get TF binding peaks
        tf_col = [col for col in base_grn_motif.columns if col.lower() == tf.lower()]
        if not tf_col:
            continue
        tf_col = tf_col[0]
        
        # All TF binding peaks genome-wide
        all_tf_peaks = base_grn_motif[base_grn_motif[tf_col] == 1].index.tolist()
        all_tf_peak_info = peak_annot[peak_annot['peak_id'].isin(all_tf_peaks)]
        
        # Peaks linked to this specific target
        target_peaks = peak_annot[
            (peak_annot['peak_id'].isin(all_tf_peaks)) &
            (peak_annot['gene_short_name'] == target)
        ]
        
        if len(target_peaks) == 0 or len(all_tf_peak_info) == 0:
            continue
        
        # Get target's chromosome
        target_chr = target_peaks['chr'].iloc[0]
        
        # How many of TF's peaks are on target's chromosome?
        tf_peaks_on_target_chr = all_tf_peak_info[all_tf_peak_info['chr'] == target_chr]
        
        proximity_data.append({
            'TF': tf,
            'target': target,
            'target_chr': target_chr,
            'tf_total_peaks': len(all_tf_peak_info),
            'tf_peaks_on_target_chr': len(tf_peaks_on_target_chr),
            'tf_peaks_linked_to_target': len(target_peaks),
            'pct_on_target_chr': len(tf_peaks_on_target_chr) / len(all_tf_peak_info) * 100,
            'pct_linked': len(target_peaks) / len(all_tf_peak_info) * 100
        })
    
    prox_df = pd.DataFrame(proximity_data)
    
    print(f"Analyzed {len(prox_df)} TF-target pairs\n")
    
    print("=== Example Cases ===\n")
    
    # Show cases where TF binds widely but only few peaks linked to target
    for _, row in prox_df.head(10).iterrows():
        print(f"{row['TF']} → {row['target']} (chr{row['target_chr']}):")
        print(f"  TF has {row['tf_total_peaks']} peaks genome-wide")
        print(f"  {row['tf_peaks_on_target_chr']} peaks on target's chromosome ({row['pct_on_target_chr']:.1f}%)")
        print(f"  {row['tf_peaks_linked_to_target']} peaks linked to this target ({row['pct_linked']:.1f}%)")
        
        if row['pct_on_target_chr'] < 20:
            print(f"  → TF binds genome-wide, target regulation is specific")
        elif row['pct_linked'] < 5:
            print(f"  → TF has many peaks on target chr, but few linked to target")
        print()
    
    print("\n=== Summary Statistics ===\n")
    print(f"Median % of TF peaks on target's chromosome: {prox_df['pct_on_target_chr'].median():.1f}%")
    print(f"Median % of TF peaks linked to specific target: {prox_df['pct_linked'].median():.1f}%")
    
    # Classify binding patterns
    print("\n=== Binding Pattern Classification ===\n")
    
    prox_df['pattern'] = 'other'
    prox_df.loc[prox_df['pct_on_target_chr'] > 80, 'pattern'] = 'chr_specific'
    prox_df.loc[prox_df['pct_on_target_chr'] < 20, 'pattern'] = 'genome_wide'
    
    print(prox_df['pattern'].value_counts())
    print()
    print("Patterns:")
    print("  chr_specific: TF binds mostly (>80%) on target's chromosome")
    print("  genome_wide: TF binds across genome (<20% on target chr)")
    print("  other: intermediate pattern")
    
    return prox_df


# Run proximity analysis
prox_df = analyze_peak_target_proximity(
    grn_df=e14_grn,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique
)

=== Peak-to-Target Proximity Analysis ===

Analyzed 11 TF-target pairs

=== Example Cases ===

E2f2 → E2f3 (chrchr13):
  TF has 13386 peaks genome-wide
  522 peaks on target's chromosome (3.9%)
  2 peaks linked to this target (0.0%)
  → TF binds genome-wide, target regulation is specific

Nhlh1 → Neurog1 (chrchr13):
  TF has 5535 peaks genome-wide
  195 peaks on target's chromosome (3.5%)
  1 peaks linked to this target (0.0%)
  → TF binds genome-wide, target regulation is specific

E2f2 → Hmga1 (chrchr17):
  TF has 13386 peaks genome-wide
  632 peaks on target's chromosome (4.7%)
  1 peaks linked to this target (0.0%)
  → TF binds genome-wide, target regulation is specific

E2f3 → Hmga1 (chrchr17):
  TF has 13051 peaks genome-wide
  623 peaks on target's chromosome (4.8%)
  1 peaks linked to this target (0.0%)
  → TF binds genome-wide, target regulation is specific

Nhlh1 → Dmrt3 (chrchr19):
  TF has 5535 peaks genome-wide
  178 peaks on target's chromosome (3.2%)
  1 peaks linked to 

In [84]:
def plot_hierarchical_network_genomic(grn_df, base_grn_motif, peak_annot, enhancer_overlap_df,
                                      stage_name, output_file, figsize=(30, 20)):
    """
    Plot hierarchical network with genomic coordinate-based layers and enhancer ratio coloring.
    """
    import matplotlib.pyplot as plt
    import networkx as nx
    import numpy as np
    import pandas as pd
    
    # Get layers and profiles
    layers, tf_profiles = assign_hierarchical_layers_genomic_hybrid(
        grn_df, base_grn_motif, peak_annot, enhancer_overlap_df
    )
    
    # Build graph for visualization (include all edges from original network)
    G = nx.from_pandas_edgelist(
        grn_df[grn_df['TF'] != grn_df['target']], 
        source='TF', 
        target='target', 
        create_using=nx.DiGraph()
    )
    
    print(f"\n=== Building {stage_name} Hierarchical Layout ===")
    print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    
    # Group nodes by layer
    layer_nodes = {}
    for node, layer in layers.items():
        if layer not in layer_nodes:
            layer_nodes[layer] = []
        layer_nodes[layer].append(node)
    
    # Calculate positions
    pos = {}
    for layer, nodes in layer_nodes.items():
        # Sort within layer by enhancer ratio (left to right: high to low)
        nodes_sorted = sorted(
            nodes, 
            key=lambda x: tf_profiles.get(x, {}).get('enhancer_like_ratio', 0), 
            reverse=True
        )
        
        n = len(nodes_sorted)
        for i, node in enumerate(nodes_sorted):
            x = (i - n/2) * 3  # Spacing between nodes
            y = -layer * 4     # Spacing between layers
            pos[node] = (x, y)
    
    # Prepare node colors based on enhancer ratio
    node_colors = []
    node_sizes = []
    
    for node in G.nodes():
        prof = tf_profiles.get(node, {})
        enh_ratio = prof.get('enhancer_like_ratio', 0)
        n_peaks = prof.get('n_peaks', 0)
        
        node_colors.append(enh_ratio)
        
        # Size based on number of peaks (log scale)
        if n_peaks > 0:
            size = 300 + np.log10(n_peaks + 1) * 300
        else:
            size = 300  # Default size for TFs without data
        node_sizes.append(size)
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Draw nodes
    nodes_collection = nx.draw_networkx_nodes(
        G, pos, 
        node_size=node_sizes,
        node_color=node_colors, 
        cmap='RdYlBu_r',  # Red (high enhancer) to Blue (low enhancer)
        vmin=0, 
        vmax=0.6,  # Scale to show differences
        alpha=0.9,
        edgecolors='black',
        linewidths=1.5
    )
    
    # Draw edges with varying transparency based on correlation
    edge_colors = []
    edge_widths = []
    
    for u, v in G.edges():
        # Get correlation from original grn_df
        edge_data = grn_df[(grn_df['TF'] == u) & (grn_df['target'] == v)]
        if len(edge_data) > 0:
            corr = edge_data['correlation'].values[0]
            # Map correlation to alpha (0.8-1.0 -> 0.2-0.8)
            alpha_val = 0.2 + (corr - 0.8) * 3  # Scale 0.8-1.0 to 0.2-0.8
            edge_colors.append((0.3, 0.3, 0.3, max(0.2, min(0.8, alpha_val))))
            edge_widths.append(1 + corr * 2)  # Thicker for higher correlation
        else:
            edge_colors.append((0.3, 0.3, 0.3, 0.3))
            edge_widths.append(1)
    
    nx.draw_networkx_edges(
        G, pos, 
        edge_color=edge_colors,
        width=edge_widths,
        arrows=True, 
        arrowsize=15,
        arrowstyle='-|>',
        connectionstyle='arc3,rad=0.1',
        node_size=node_sizes
    )
    
    # Draw labels
    # Adjust font size based on number of nodes
    if G.number_of_nodes() > 40:
        font_size = 8
    elif G.number_of_nodes() > 25:
        font_size = 10
    else:
        font_size = 12
    
    # Add labels with background for readability
    for node, (x, y) in pos.items():
        prof = tf_profiles.get(node, {})
        enh_ratio = prof.get('enhancer_like_ratio', 0)
        n_peaks = prof.get('n_peaks', 0)
        
        # Label with enhancer percentage
        if n_peaks > 0:
            label = f"{node}\n{enh_ratio:.0%}"
        else:
            label = f"{node}\n(no data)"
        
        ax.text(
            x, y, label,
            fontsize=font_size,
            fontweight='bold',
            ha='center',
            va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                     edgecolor='none', alpha=0.8)
        )
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(
        cmap='RdYlBu_r', 
        norm=plt.Normalize(vmin=0, vmax=0.6)
    )
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('Enhancer-like Binding Ratio', rotation=270, labelpad=25, fontsize=14)
    cbar.set_ticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
    cbar.set_ticklabels(['0%', '10%', '20%', '30%', '40%', '50%', '60%'])
    
    # Add layer labels on the left
    for layer in sorted(layer_nodes.keys()):
        nodes = layer_nodes[layer]
        if nodes:
            y_pos = -layer * 4
            ax.text(
                min([pos[n][0] for n in nodes]) - 5, y_pos,
                f'Layer {layer}',
                fontsize=14,
                fontweight='bold',
                va='center',
                ha='right',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.7)
            )
    
    # Title and formatting
    title = f'{stage_name} TF Regulatory Network - Genomic Hierarchy\n'
    title += f'({G.number_of_nodes()} TFs, {G.number_of_edges()} regulatory links)\n'
    title += f'Color: Enhancer binding ratio | Size: Number of binding peaks | Edge: Correlation strength'
    
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.axis('off')
    plt.tight_layout()
    
    # Save
    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\nSaved to: {output_file}")
    plt.show()
    
    # Print summary statistics
    print(f"\n=== {stage_name} Network Statistics ===")
    print(f"Total TFs: {G.number_of_nodes()}")
    print(f"Total edges: {G.number_of_edges()}")
    print(f"Layers: {len(layer_nodes)}")
    
    # Calculate average out-degree per layer
    print(f"\n=== Layer Statistics ===")
    for layer in sorted(layer_nodes.keys()):
        nodes = layer_nodes[layer]
        out_degrees = [G.out_degree(n) for n in nodes]
        in_degrees = [G.in_degree(n) for n in nodes]
        avg_enh = np.mean([tf_profiles.get(n, {}).get('enhancer_like_ratio', 0) 
                          for n in nodes if tf_profiles.get(n, {}).get('n_peaks', 0) > 0])
        
        print(f"\nLayer {layer}: {len(nodes)} TFs")
        print(f"  Avg out-degree: {np.mean(out_degrees):.1f}")
        print(f"  Avg in-degree: {np.mean(in_degrees):.1f}")
        print(f"  Avg enhancer ratio: {avg_enh:.1%}")
        print(f"  Hub TFs (out-degree > 5): {[n for n in nodes if G.out_degree(n) > 5]}")
    
    return G, layers, tf_profiles, pos


# Plot E14 network
G_e14, layers_e14, profiles_e14, pos_e14 = plot_hierarchical_network_genomic(
    grn_df=e14_grn,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    enhancer_overlap_df=overlaps_df,
    stage_name='E14',
    output_file='e14_genomic_hierarchy.png',
    figsize=(30, 20)
)

# Plot E18 network
G_e18, layers_e18, profiles_e18, pos_e18 = plot_hierarchical_network_genomic(
    grn_df=e18_grn,
    base_grn_motif=base_grn_motif_indexed,
    peak_annot=peak_annot_unique,
    enhancer_overlap_df=overlaps_df,
    stage_name='E18',
    output_file='e18_genomic_hierarchy.png',
    figsize=(30, 20)
)


=== Peak Classification (Hybrid) ===
Total unique peaks: 17436
EnhancerDB: 6309 (36.2%)
Promoter: 8565 (49.1%)
Combined enhancer-like: 7817 (44.8%)

=== TF Binding Profiles ===
Dmrt2: 109 peaks → 50 enh-like (45.9%), median_dist=275bp
Dmrt3: 69 peaks → 26 enh-like (37.7%), median_dist=246bp
E2f2: 13386 peaks → 6615 enh-like (49.4%), median_dist=200bp
E2f3: 13051 peaks → 6494 enh-like (49.8%), median_dist=202bp
Hmga1: 109 peaks → 47 enh-like (43.1%), median_dist=313bp
Lef1: 205 peaks → 102 enh-like (49.8%), median_dist=288bp
Neurog1: 1297 peaks → 623 enh-like (48.0%), median_dist=251bp
Nhlh1: 5535 peaks → 2724 enh-like (49.2%), median_dist=234bp
Nr0b1: 1370 peaks → 756 enh-like (55.2%), median_dist=287bp
Plagl2: NOT FOUND in motif matrix
Sall4: 4213 peaks → 2230 enh-like (52.9%), median_dist=202bp
Tcf7l1: 184 peaks → 93 enh-like (50.5%), median_dist=307bp
Zfp599: NOT FOUND in motif matrix

=== Cycle Breaking ===
Removed 1 edges:
  E2f2 (49.4% enh-like) → E2f3 (49.8% enh-like)

=== Laye

/home/users/adhal/micromamba/envs/scrna_target_idf/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


# ==============================================================================
# 5. TRIPLET SYNERGY USING O Information
# ==============================================================================

In [88]:
"""
Greedy O-information maximization starting with triplets.
"""

import numpy as np
import pandas as pd
from scipy.stats import entropy
from itertools import combinations
from tqdm import tqdm
from sklearn.metrics import mutual_info_score


class GreedyTFCombinationFinder:
    """Find optimal TF combinations starting with triplets."""
    
    def __init__(self, grn_df, counts_file, metadata_file, deseq_file, stage, n_bins=3):
        """
        Args:
            grn_df: GRN DataFrame with [TF, target, ...]
            counts_file, metadata_file, deseq_file: Data files
            stage: 'E14' or 'E18'
            n_bins: Number of bins for discretization
        """
        self.grn = grn_df
        self.stage = stage
        self.n_bins = n_bins
        
        # Load expression data
        counts = pd.read_csv(counts_file, sep=';', header=0).iloc[:, 1:]
        counts = counts.set_index(counts.columns[0])
        
        # Map to symbols
        deseq = pd.read_csv(deseq_file, index_col=0)
        ensembl_to_symbol = dict(zip(deseq['gene_id'], deseq['symbol']))
        counts.index = counts.index.map(lambda x: ensembl_to_symbol.get(x, x))
        counts = counts[~counts.index.duplicated(keep='first')]
        
        metadata = pd.read_csv(metadata_file, sep=';')
        
        # Get stage samples
        mask = (metadata['Stage'] == stage) & (metadata['Marker'] == 'Progenitors')
        samples = metadata[mask]['SampleID'].tolist()
        
        # Normalize
        expr = counts[samples]
        cpm = expr.div(expr.sum(axis=0), axis=1) * 1e6
        self.expr = np.log2(cpm + 1)
        
        print(f"Loaded expression for {stage}: {self.expr.shape[0]} genes, {self.expr.shape[1]} samples")
    
    def _discretize(self, x):
        """Discretize continuous values into bins."""
        return pd.qcut(x, q=self.n_bins, labels=False, duplicates='drop')
    
    def _multi_information(self, *variables):
        """
        Calculate multi-information (total correlation) for n variables.
        MI(X1, X2, ..., Xn) = sum(H(Xi)) - H(X1, X2, ..., Xn)
        """
        # Individual entropies
        individual_entropies = 0
        for var in variables:
            var_disc = self._discretize(var)
            counts = pd.Series(var_disc).value_counts()
            probs = counts / counts.sum()
            individual_entropies += entropy(probs, base=2)
        
        # Joint entropy
        discretized = [self._discretize(var) for var in variables]
        joint_df = pd.DataFrame({f'v{i}': discretized[i] for i in range(len(variables))})
        joint_df['joint'] = joint_df.astype(str).agg('_'.join, axis=1)
        
        counts = joint_df['joint'].value_counts()
        probs = counts / counts.sum()
        joint_entropy = entropy(probs, base=2)
        
        multi_info = individual_entropies - joint_entropy
        return multi_info
    
    def _o_information(self, *variables):
        """
        Calculate O-information for n variables.
        
        For triplets: O = -I(X;Y;Z)
        
        Where I(X;Y;Z) = interaction information = I(X;Y) + I(X;Z) + I(Y;Z) - I(X,Y;Z)
        
        Interpretation:
        I(X;Y;Z) > 0 → Synergy → O < 0
        I(X;Y;Z) < 0 → Redundancy → O > 0
        
        BUT we want intuitive signs, so return I directly:
        I > 0: Synergy (TFs cooperate)
        I < 0: Redundancy (TFs overlap)
        """
        n = len(variables)
        
        if n == 3:
            x, y, z = variables
            
            # Pairwise MIs
            mi_xy = mutual_info_score(self._discretize(x), self._discretize(y))
            mi_xz = mutual_info_score(self._discretize(x), self._discretize(z))
            mi_yz = mutual_info_score(self._discretize(y), self._discretize(z))
            
            # Joint MI I(X,Y;Z)
            x_disc = self._discretize(x)
            y_disc = self._discretize(y)
            z_disc = self._discretize(z)
            
            x_str = pd.Series(x_disc).astype(str)
            y_str = pd.Series(y_disc).astype(str)
            xy_joint = x_str + '_' + y_str
            xy_mapping = {v: i for i, v in enumerate(xy_joint.unique())}
            xy_numeric = xy_joint.map(xy_mapping).values
            
            mi_xyz = mutual_info_score(xy_numeric, z_disc)
            
            # Interaction information (synergy measure)
            interaction_info = mi_xy + mi_xz + mi_yz - mi_xyz
            
            tc = self._multi_information(*variables)
            
            # Return I directly (positive = synergy)
            return interaction_info, tc

    def _mutual_info_raw(self, x, y):
        """Calculate MI without using sklearn (for consistency)."""
        x_disc = self._discretize(x)
        y_disc = self._discretize(y)
        return mutual_info_score(x_disc, y_disc)
    
    def find_top_triplets(self, min_shared_targets=1, top_n=10):
        """
        Find top TF triplets (TF1, TF2, Target) by O-information.
        
        Returns:
            DataFrame with top triplets
        """
        print(f"\n=== Finding Top TF Triplets ===")
        
        # Get TF-target relationships
        tf_targets = {}
        for _, row in self.grn.iterrows():
            tf = row['TF']
            target = row['target']
            if tf not in tf_targets:
                tf_targets[tf] = set()
            tf_targets[tf].add(target)
        
        self.tf_targets = tf_targets
        
        # Find TF pairs with shared targets
        results = []
        tfs = [tf for tf in tf_targets.keys() if tf in self.expr.index]
        
        print(f"Evaluating {len(list(combinations(tfs, 2)))} TF pairs...")
        
        for tf1, tf2 in tqdm(combinations(tfs, 2), desc="Finding triplets", total=len(list(combinations(tfs, 2)))):
            shared_targets = tf_targets[tf1] & tf_targets[tf2]
            if len(shared_targets) < min_shared_targets:
                continue
            
            # Evaluate each shared target
            for target in shared_targets:
                if target not in self.expr.index:
                    continue
                
                tf1_expr = self.expr.loc[tf1].values
                tf2_expr = self.expr.loc[tf2].values
                target_expr = self.expr.loc[target].values
                
                try:
                    o_info, tc = self._o_information(tf1_expr, tf2_expr, target_expr)
                    
                    results.append({
                        'TF1': tf1,
                        'TF2': tf2,
                        'target': target,
                        'o_info': o_info,
                        'tc': tc
                    })
                except Exception as e:
                    continue
        
        triplets_df = pd.DataFrame(results).sort_values('o_info', ascending=False)
        
        print(f"Found {len(triplets_df)} valid triplets")
        print(f"\nTop {top_n} triplets:")
        print(triplets_df.head(top_n))
        
        return triplets_df.head(top_n)
    
    def greedy_expansion(self, initial_triplet, remaining_tfs, max_order=5):
        """
        Greedily add TFs to an initial triplet.
        
        Args:
            initial_triplet: Dict with keys TF1, TF2, target
            remaining_tfs: List of TFs to try adding
            max_order: Maximum combination order (including target)
        
        Returns:
            List of results for each expansion step
        """
        tf1 = initial_triplet['TF1']
        tf2 = initial_triplet['TF2']
        target = initial_triplet['target']
        
        current_tfs = [tf1, tf2]
        
        results = []
        
        # Baseline: order-3 (TF1, TF2, Target)
        exprs = [self.expr.loc[tf1].values, self.expr.loc[tf2].values, self.expr.loc[target].values]
        baseline, tc = self._o_information(*exprs)
        
        results.append({
            'order': 3,
            'tfs': current_tfs.copy(),
            'target': target,
            'o_info': baseline,
            'tc': tc
        })
        
        print(f"\n  Baseline ({tf1}, {tf2}, {target}): O-info = {baseline:.4f}")
        
        # Iteratively add TFs
        for order in range(4, max_order + 1):
            best_tf = None
            best_o_info = baseline
            best_tc = tc
            
            # Try adding each remaining TF
            for candidate_tf in remaining_tfs:
                if candidate_tf in current_tfs:
                    continue
                if candidate_tf not in self.expr.index:
                    continue
                
                # Check if candidate regulates the target
                if target not in self.tf_targets.get(candidate_tf, set()):
                    continue
                
                # Calculate O-info with candidate
                exprs = [self.expr.loc[tf].values for tf in current_tfs + [candidate_tf]] + [self.expr.loc[target].values]
                
                try:
                    candidate_o_info, candidate_tc = self._o_information(*exprs)
                except:
                    continue
                
                if candidate_o_info > best_o_info:
                    best_o_info = candidate_o_info
                    best_tf = candidate_tf
                    best_tc = candidate_tc
            
            # Stop if no improvement
            if best_tf is None or best_o_info <= baseline:
                print(f"  Order {order}: No improvement, stopping")
                break
            
            # Add best TF
            current_tfs.append(best_tf)
            baseline = best_o_info
            tc = best_tc
            
            results.append({
                'order': order,
                'tfs': current_tfs.copy(),
                'target': target,
                'o_info': best_o_info,
                'tc': best_tc
            })
            
            print(f"  Order {order}: Added {best_tf}, O-info = {best_o_info:.4f}")
        
        return results
    
    def find_optimal_combinations(self, top_triplets=10, max_order=6):
        """
        Main pipeline: find top triplets, then greedily expand each.
        
        Returns:
            DataFrame with all expansion results
        """
        # Step 1: Find top triplets
        triplets_df = self.find_top_triplets(top_n=top_triplets)
        
        if len(triplets_df) == 0:
            print("No valid triplets found!")
            return pd.DataFrame()
        
        # Step 2: Get all TFs
        all_tfs = list(self.tf_targets.keys())
        
        # Step 3: Expand each top triplet
        all_results = []
        
        print(f"\n=== Greedy Expansion ===")
        for idx, row in triplets_df.iterrows():
            initial_triplet = {
                'TF1': row['TF1'],
                'TF2': row['TF2'],
                'target': row['target']
            }
            
            remaining_tfs = [tf for tf in all_tfs if tf not in [row['TF1'], row['TF2']]]
            
            print(f"\nExpanding triplet {idx+1}/{len(triplets_df)}: ({row['TF1']}, {row['TF2']}, {row['target']})")
            
            expansion = self.greedy_expansion(initial_triplet, remaining_tfs, max_order=max_order)
            
            for result in expansion:
                result['initial_triplet'] = f"{row['TF1']}+{row['TF2']}→{row['target']}"
                result['stage'] = self.stage
                all_results.append(result)
        
        results_df = pd.DataFrame(all_results)
        
        # Summary
        print(f"\n=== Summary ===")
        print(f"Total combinations found: {len(results_df)}")
        print(f"\nBest combinations by order:")
        for order in sorted(results_df['order'].unique()):
            best = results_df[results_df['order'] == order].sort_values('o_info', ascending=False).iloc[0]
            print(f"  Order {order}: {best['tfs']} → {best['target']} - O-info = {best['o_info']:.4f}")
        
        return results_df


# === USAGE ===
# e14_finder = GreedyTFCombinationFinder(
#     grn_df=e14_grn,
#     counts_file=paths['counts'],
#     metadata_file=paths['metadata'],
#     deseq_file=paths['deseq_output'],
#     stage='E14',
#     n_bins=3
# )
# 
# e14_combinations = e14_finder.find_optimal_combinations(top_triplets=10, max_order=6)
# e14_combinations.to_csv('e14_optimal_tf_combinations.csv', index=False)

In [92]:

# === USAGE ===
e14_finder = GreedyTFCombinationFinder(
    grn_df=e14_grn,
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    stage='E14',
    n_bins=3
)

e14_combinations = e14_finder.find_optimal_combinations(top_triplets=10, max_order=5)
e14_combinations.to_csv('/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/e14_optimal_tf_combinations.csv', index=False)

# E18
e18_finder = GreedyTFCombinationFinder(
    grn_df=e18_grn,
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    stage='E18',
    n_bins=3
)

e18_combinations = e18_finder.find_optimal_combinations(top_triplets=10, max_order=5)
e18_combinations.to_csv('/mnt/lscratch/users/adhal/CorticalNeuronFate/CellConversionNSC/results/cd133_e14_e18_grn/e18_optimal_tf_combinations.csv', index=False)

Loaded expression for E14: 28656 genes, 10 samples

=== Finding Top TF Triplets ===
Evaluating 55 TF pairs...


Finding triplets: 100%|██████████| 55/55 [00:01<00:00, 40.99it/s]


Found 48 valid triplets

Top 10 triplets:
        TF1      TF2  target    o_info        tc
46     Rorc    Nr4a2  Zfp599  1.555323  2.243856
45    Sall4    Nr4a2   Rcor2  1.555323  2.243856
2      E2f2     E2f3   Hmga1  1.346023  2.541901
1      E2f2     E2f3  Plagl2  1.346023  2.541901
22    Nhlh1  Neurog1   Rcor2  1.259719  2.266412
3      E2f2     E2f3   Hmga2  1.207394  2.066412
25    Nhlh2   Bcl11b    Six4  1.121089  1.790924
47  Neurog1    Nr4a2   Rcor2  0.930135  2.017390
44    Sall4  Neurog1   Rcor2  0.930135  2.017390
20    Nhlh1    Sall4   Dmrt3  0.896155  2.217390

=== Greedy Expansion ===

Expanding triplet 47/10: (Rorc, Nr4a2, Zfp599)

  Baseline (Rorc, Nr4a2, Zfp599): O-info = 1.5553
  Order 4: No improvement, stopping

Expanding triplet 46/10: (Sall4, Nr4a2, Rcor2)

  Baseline (Sall4, Nr4a2, Rcor2): O-info = 1.5553
  Order 4: No improvement, stopping

Expanding triplet 3/10: (E2f2, E2f3, Hmga1)

  Baseline (E2f2, E2f3, Hmga1): O-info = 1.3460
  Order 4: No improvement, st

Finding triplets: 100%|██████████| 153/153 [00:01<00:00, 87.28it/s] 


Found 62 valid triplets

Top 10 triplets:
      TF1    TF2 target    o_info        tc
42  Foxf2  Foxq1   Etv4  1.761912  2.541901
37  Nr4a1    Fos  Ppara  1.536978  2.541901
61   Hey2  Rreb1   Rora  1.484653  2.341901
59   Gli1   Hey2  Rreb1  1.346023  2.492879
6   Klf15   Hey2   Rora  1.309333  2.190924
33   Klf9   Hey2   Rora  1.309333  2.190924
34   Klf9   Hey2  Foxo1  1.309333  2.190924
35   Klf9  Rreb1   Rora  1.241374  2.141901
8   Klf15  Rreb1   Rora  1.241374  2.141901
2   Klf15   Klf9   Rora  1.170704  2.390924

=== Greedy Expansion ===

Expanding triplet 43/10: (Foxf2, Foxq1, Etv4)

  Baseline (Foxf2, Foxq1, Etv4): O-info = 1.7619
  Order 4: No improvement, stopping

Expanding triplet 38/10: (Nr4a1, Fos, Ppara)

  Baseline (Nr4a1, Fos, Ppara): O-info = 1.5370
  Order 4: No improvement, stopping

Expanding triplet 62/10: (Hey2, Rreb1, Rora)

  Baseline (Hey2, Rreb1, Rora): O-info = 1.4847
  Order 4: No improvement, stopping

Expanding triplet 60/10: (Gli1, Hey2, Rreb1)

  Base

In [90]:
e14_combinations


,order,tfs,target,o_info,tc,initial_triplet,stage
0,3,"[Rorc, Nr4a2]",Zfp599,1.555323,2.243856,Rorc+Nr4a2→Zfp599,E14
1,3,"[Sall4, Nr4a2]",Rcor2,1.555323,2.243856,Sall4+Nr4a2→Rcor2,E14
2,3,"[E2f2, E2f3]",Hmga1,1.346023,2.541901,E2f2+E2f3→Hmga1,E14
3,3,"[E2f2, E2f3]",Plagl2,1.346023,2.541901,E2f2+E2f3→Plagl2,E14
4,3,"[Nhlh1, Neurog1]",Rcor2,1.259719,2.266412,Nhlh1+Neurog1→Rcor2,E14
5,3,"[E2f2, E2f3]",Hmga2,1.207394,2.066412,E2f2+E2f3→Hmga2,E14
6,3,"[Nhlh2, Bcl11b]",Six4,1.121089,1.790924,Nhlh2+Bcl11b→Six4,E14
7,3,"[Neurog1, Nr4a2]",Rcor2,0.930135,2.017390,Neurog1+Nr4a2→Rcor2,E14
8,3,"[Sall4, Neurog1]",Rcor2,0.930135,2.017390,Sall4+Neurog1→Rcor2,E14
9,3,"[Nhlh1, Sall4]",Dmrt3,0.896155,2.217390,Nhlh1+Sall4→Dmrt3,E14


In [91]:
e18_combinations

,order,tfs,target,o_info,tc,initial_triplet,stage
0,3,"[Foxf2, Foxq1]",Etv4,1.761912,2.541901,Foxf2+Foxq1→Etv4,E18
1,3,"[Nr4a1, Fos]",Ppara,1.536978,2.541901,Nr4a1+Fos→Ppara,E18
2,3,"[Hey2, Rreb1]",Rora,1.484653,2.341901,Hey2+Rreb1→Rora,E18
3,3,"[Gli1, Hey2]",Rreb1,1.346023,2.492879,Gli1+Hey2→Rreb1,E18
4,3,"[Klf15, Hey2]",Rora,1.309333,2.190924,Klf15+Hey2→Rora,E18
5,3,"[Klf9, Hey2]",Rora,1.309333,2.190924,Klf9+Hey2→Rora,E18
6,3,"[Klf9, Hey2]",Foxo1,1.309333,2.190924,Klf9+Hey2→Foxo1,E18
7,3,"[Klf9, Rreb1]",Rora,1.241374,2.141901,Klf9+Rreb1→Rora,E18
8,3,"[Klf15, Rreb1]",Rora,1.241374,2.141901,Klf15+Rreb1→Rora,E18
9,3,"[Klf15, Klf9]",Rora,1.170704,2.390924,Klf15+Klf9→Rora,E18
